# 신경망으로 XOR문제 풀어보기

In [ ]:
import numpy as np
import tensorflow as tf

X = tf.Variable([[0,0],[0,1],[1,0],[1,1]])

신경망 생성

In [ ]:
# 유닛 함수 생성
def unit(X, W, b):
    z = tf.matmul(X, W) + b  
    hypothesis = tf.cast(z > 0, 1, 0)  # 아마도 tanh?
    return hypothesis

In [ ]:
# 유닛 1
W1 = tf.Variable([[5],[5]])
b1 = tf.Variable([-8])

hypothesis1 = unit(X, W1, b1)
hypothesis1

<tf.Tensor: shape=(4, 1), dtype=float32, numpy=
array([[0.],
       [0.],
       [0.],
       [1.]], dtype=float32)>

In [ ]:
# 유닛 2
W2 = tf.Variable([[-7],[-7]])
b2 = tf.Variable([3])

hypothesis2 = unit(X, W2, b2)
hypothesis2

<tf.Tensor: shape=(4, 1), dtype=float32, numpy=
array([[1.],
       [0.],
       [0.],
       [0.]], dtype=float32)>

In [ ]:
# 유닛 3
X2 = tf.cast(tf.concat([hypothesis1, hypothesis2], 1), 'int32')  # 유닛 1,2의 출력을 입력으로 받는다. 
W3 = tf.Variable([[-11],[-11]])
b3 = tf.Variable([6])

y_hat = unit(X2, W3, b3)
y_hat

<tf.Tensor: shape=(4, 1), dtype=float32, numpy=
array([[0.],
       [1.],
       [1.],
       [0.]], dtype=float32)>

실제로 작동이 된다.

## 조금 다른 버전

In [ ]:
import numpy as np
import tensorflow as tf

x_data = [[0, 0], [0, 1], [1, 0], [1, 1]]
y_data = [[0], [1], [1], [0]]

dataset = tf.data.Dataset.from_tensor_slices((x_data, y_data)).batch(len(x_data)) # 데이터 셋 

def preprocess_data(features, labels):
    features = tf.cast(features, tf.float32)
    labels = tf.cast(labels, tf.float32)
    return features, labels

tf.random.set_seed(1)
# 1번 레이어 유닛 1
W1 = tf.Variable(tf.random.normal([2, 1]))
b1 = tf.Variable(tf.random.normal([1]))
# 1번 레이어 유닛 2                      
W2 = tf.Variable(tf.random.normal([2, 1]))
b2 = tf.Variable(tf.random.normal([1]))
# 2번 레이어 유닛 1
W3 = tf.Variable(tf.random.normal([2, 1]))
b3 = tf.Variable(tf.random.normal([1]))

def neural_net(features):
    layer1 = tf.sigmoid(tf.matmul(features, W1) + b1)
    layer2 = tf.sigmoid(tf.matmul(features, W2) + b2)
    layer3 = tf.concat([layer1, layer2], axis=1)
    layer3 = tf.reshape(layer3, shape=[-1, 2])
    hypothesis = tf.sigmoid(tf.matmul(layer3, W3) + b3)
    return hypothesis

def loss_fn(hypothesis, lables):
    cost = -tf.reduce_mean(labels * tf.math.log(hypothesis) + (1 - labels) * tf.math.log(1 - hypothesis))
    return cost

optimizer = tf.keras.optimizers.SGD(learning_rate=0.01)

def accuracy_fn(hypothesis, labels):
    predicted = tf.cast(hypothesis > 0.5, tf.float32)
    accuracy = tf.reduce_mean(tf.cast(tf.equal(predicted, labels), tf.float32))
    return accuracy

def grad(features, labels):
    with tf.GradientTape() as tape:
        loss_value = loss_fn(neural_net(features), labels)
    return tape.gradient(loss_value, [W1, W2, W3, b1, b2, b3])

EPOCHS = 5000
for step in range(EPOCHS):
    for features, labels in dataset:
        features, labels = preprocess_data(features, labels)
        grads, cost = grad(neural_net(features), labels)
        optimizer.apply_gradients(zip(grads, [W1, W2, W3, b1, b2, b3]))
        if step % 500 == 0:
            print(f"{step} {cost}")

x_data, y_data = preprocess_data(x_data, y_data)
test_acc = accuracy_fn(neural_net(x_data), y_data)
print(f"Testset Accuracy: {test_acc}")

# 모델 사용하기

In [ ]:
# Lab 9 XOR
import tensorflow as tf
from tensorflow import keras
import numpy as np

x_data = np.array([[0, 0], [0, 1], [1, 0], [1, 1]], dtype=np.float32)
y_data = np.array([[0], [1], [1], [0]], dtype=np.float32)

In [ ]:
model = keras.Sequential(
    keras.layers.Dense(units=2, input_dim=2,activation='sigmoid')
)
model.add(keras.layers.Dense(units=1, activation='sigmoid'))
model.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 dense (Dense)               (None, 2)                 6         
                                                                 
 dense_1 (Dense)             (None, 1)                 3         
                                                                 
Total params: 9
Trainable params: 9
Non-trainable params: 0
_________________________________________________________________


In [ ]:
model.compile(loss='binary_crossentropy', 
              optimizer=tf.optimizers.SGD(lr=0.1), 
              metrics=['accuracy'])
model.summary()

Model: "sequential_5"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 dense_26 (Dense)            (None, 2)                 6         
                                                                 
 dense_27 (Dense)            (None, 1)                 3         
                                                                 
Total params: 9
Trainable params: 9
Non-trainable params: 0
_________________________________________________________________


/usr/local/lib/python3.7/dist-packages/keras/optimizer_v2/gradient_descent.py:102: UserWarning: The `lr` argument is deprecated, use `learning_rate` instead.
  super(SGD, self).__init__(name, **kwargs)


In [ ]:
history = model.fit(x_data, y_data, epochs=10000)

스트리밍 출력 내용이 길어서 마지막 5000줄이 삭제되었습니다.
1/1 [==============================] - 0s 9ms/step - loss: 0.3547 - accuracy: 0.5000
Epoch 7502/10000
1/1 [==============================] - 0s 18ms/step - loss: 0.3547 - accuracy: 0.5000
Epoch 7503/10000
1/1 [==============================] - 0s 10ms/step - loss: 0.3547 - accuracy: 0.5000
Epoch 7504/10000
1/1 [==============================] - 0s 6ms/step - loss: 0.3547 - accuracy: 0.5000
Epoch 7505/10000
1/1 [==============================] - 0s 6ms/step - loss: 0.3547 - accuracy: 0.5000
Epoch 7506/10000
1/1 [==============================] - 0s 6ms/step - loss: 0.3547 - accuracy: 0.5000
Epoch 7507/10000
1/1 [==============================] - 0s 6ms/step - loss: 0.3547 - accuracy: 0.5000
Epoch 7508/10000
1/1 [==============================] - 0s 7ms/step - loss: 0.3547 - accuracy: 0.5000
Epoch 7509/10000
1/1 [==============================] - 0s 6ms/step - loss: 0.3547 - accuracy: 0.5000
Epoch 7510/10000
1/1 [==============================] - 0s 10

In [ ]:
predictions = model.predict(x_data)
tf.cast(predictions > 0.5, 'float32')

<tf.Tensor: shape=(4, 1), dtype=float32, numpy=
array([[0.],
       [1.],
       [0.],
       [1.]], dtype=float32)>

In [ ]:
score = model.evaluate(x_data, y_data)
score

1/1 [==============================] - 0s 24ms/step - loss: 0.3526 - accuracy: 0.5000


[0.35260024666786194, 0.5]

# 여러가지 해보기

In [ ]:
import tensorflow as tf
from tensorflow import keras
import numpy as np

x_data = np.array([[0, 0], [0, 1], [1, 0], [1, 1]], dtype=np.float32)
y_data = np.array([[0], [1], [1], [0]], dtype=np.float32)

In [ ]:
tf.random.set_seed(1)

W1 = tf.Variable(tf.random.normal([2, 2]))  # (입력, 출력) 출력이 다음 레이어의 입력이다.
b1 = tf.Variable(tf.random.normal([2]))    

W2 = tf.Variable(tf.random.normal([2, 1])) 
b2 = tf.Variable(tf.random.normal([1]))

learning_rate = 0.1
for i in range(3001):
    with tf.GradientTape() as tape:
        layer1 = tf.sigmoid(tf.matmul(x_data, W1) + b1)       # 레이어 1
        hypothesis = tf.sigmoid(tf.matmul(layer1, W2) + b2)   # 레이어 2
        cost = -tf.reduce_mean(y_data * tf.math.log(hypothesis) + (1 - y_data) * tf.math.log(1 - hypothesis))

    W1_grad, b1_grad, W2_grad, b2_grad = tape.gradient(cost, [W1, b1, W2, b2])

    W1.assign_sub(learning_rate * W1_grad)
    b1.assign_sub(learning_rate * b1_grad)
    W2.assign_sub(learning_rate * W2_grad)
    b2.assign_sub(learning_rate * b2_grad)

    predicted = tf.cast(hypothesis > 0.5, tf.float32)
    accuracy = tf.reduce_mean(tf.cast(tf.equal(predicted, y_data), tf.float32))

    if i % 300 == 0:
        print(f"{i} \n{hypothesis} \n{cost} \n{predicted} \n{accuracy}")

0 
[[0.7887727]
 [0.790856 ]
 [0.7846526]
 [0.792491 ]] 
0.9011387228965759 
[[1.]
 [1.]
 [1.]
 [1.]] 
0.5
300 
[[0.50329244]
 [0.48733142]
 [0.5100053 ]
 [0.48818466]] 
0.690422534942627 
[[1.]
 [0.]
 [1.]
 [0.]] 
0.5
600 
[[0.505    ]
 [0.4767793]
 [0.5241018]
 [0.4751736]] 
0.6836640238761902 
[[1.]
 [0.]
 [1.]
 [0.]] 
0.5
900 
[[0.5024481 ]
 [0.4647629 ]
 [0.54295135]
 [0.46085873]] 
0.6731990575790405 
[[1.]
 [0.]
 [1.]
 [0.]] 
0.5
1200 
[[0.4934143 ]
 [0.44981024]
 [0.5730443 ]
 [0.44068888]] 
0.6542081832885742 
[[0.]
 [0.]
 [1.]
 [0.]] 
0.75
1500 
[[0.476364  ]
 [0.4367831 ]
 [0.61600137]
 [0.41019475]] 
0.621936559677124 
[[0.]
 [0.]
 [1.]
 [0.]] 
0.75
1800 
[[0.45018178]
 [0.43873942]
 [0.6623417 ]
 [0.36847937]] 
0.5734038352966309 
[[0.]
 [0.]
 [1.]
 [0.]] 
0.75
2100 
[[0.4089204 ]
 [0.4728482 ]
 [0.69933814]
 [0.3181747 ]] 
0.503847062587738 
[[0.]
 [0.]
 [1.]
 [0.]] 
0.75
2400 
[[0.3449961 ]
 [0.5486351 ]
 [0.7275551 ]
 [0.26399568]] 
0.41200512647628784 
[[0.]
 [1.]
 [1.

## 넓은 NN 해보기

2번 레이어의 유닛을 10개로 만들어본다.

In [ ]:
tf.random.set_seed(1)

W1 = tf.Variable(tf.random.normal([2, 10]))  # (입력, 출력) 출력이 다음 레이어의 입력이다.
b1 = tf.Variable(tf.random.normal([10]))    

W2 = tf.Variable(tf.random.normal([10, 1])) 
b2 = tf.Variable(tf.random.normal([1]))

learning_rate = 0.1
for i in range(3001):
    with tf.GradientTape() as tape:
        layer1 = tf.sigmoid(tf.matmul(x_data, W1) + b1)       # 레이어 1
        hypothesis = tf.sigmoid(tf.matmul(layer1, W2) + b2)   # 레이어 2
        cost = -tf.reduce_mean(y_data * tf.math.log(hypothesis) + (1 - y_data) * tf.math.log(1 - hypothesis))

    W1_grad, b1_grad, W2_grad, b2_grad = tape.gradient(cost, [W1, b1, W2, b2])

    W1.assign_sub(learning_rate * W1_grad)
    b1.assign_sub(learning_rate * b1_grad)
    W2.assign_sub(learning_rate * W2_grad)
    b2.assign_sub(learning_rate * b2_grad)

    predicted = tf.cast(hypothesis > 0.5, tf.float32)
    accuracy = tf.reduce_mean(tf.cast(tf.equal(predicted, y_data), tf.float32))

    if i % 300 == 0:
        print(f"{i} \n{hypothesis} \n{cost} \n{predicted} \n{accuracy}")

0 
[[0.67343587]
 [0.6112162 ]
 [0.66067773]
 [0.59812456]] 
0.7343839406967163 
[[1.]
 [1.]
 [1.]
 [1.]] 
0.5
300 
[[0.50434095]
 [0.5098052 ]
 [0.51762366]
 [0.47257644]] 
0.668462872505188 
[[1.]
 [1.]
 [1.]
 [0.]] 
0.75
600 
[[0.45236224]
 [0.5720483 ]
 [0.5126673 ]
 [0.46280575]] 
0.6125491857528687 
[[0.]
 [1.]
 [1.]
 [0.]] 
1.0
900 
[[0.36802968]
 [0.659204  ]
 [0.53706753]
 [0.42632335]] 
0.5132390260696411 
[[0.]
 [1.]
 [1.]
 [0.]] 
1.0
1200 
[[0.26791027]
 [0.7448498 ]
 [0.6066366 ]
 [0.36074623]] 
0.3884260058403015 
[[0.]
 [1.]
 [1.]
 [0.]] 
1.0
1500 
[[0.18530166]
 [0.80992174]
 [0.70424056]
 [0.27753177]] 
0.2728680372238159 
[[0.]
 [1.]
 [1.]
 [0.]] 
1.0
1800 
[[0.12882265]
 [0.8581518 ]
 [0.7912859 ]
 [0.20140214]] 
0.18746943771839142 
[[0.]
 [1.]
 [1.]
 [0.]] 
1.0
2100 
[[0.09207842]
 [0.89340097]
 [0.8522754 ]
 [0.14594372]] 
0.13173021376132965 
[[0.]
 [1.]
 [1.]
 [0.]] 
1.0
2400 
[[0.06821171]
 [0.91819745]
 [0.8919267 ]
 [0.10883246]] 
0.09639666974544525 
[[0.]
 

- 모델로 넓게 해보기

In [ ]:
model = keras.Sequential(
    keras.layers.Dense(units=10, input_dim=2,activation='sigmoid')
)
model.add(keras.layers.Dense(units=1, activation='sigmoid'))

model.compile(loss='binary_crossentropy', 
              optimizer=tf.optimizers.SGD(lr=0.1), 
              metrics=['accuracy'])
model.summary()

Model: "sequential_1"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 dense_2 (Dense)             (None, 10)                30        
                                                                 
 dense_3 (Dense)             (None, 1)                 11        
                                                                 
Total params: 41
Trainable params: 41
Non-trainable params: 0
_________________________________________________________________


/usr/local/lib/python3.7/dist-packages/keras/optimizer_v2/gradient_descent.py:102: UserWarning: The `lr` argument is deprecated, use `learning_rate` instead.
  super(SGD, self).__init__(name, **kwargs)


In [ ]:
history = model.fit(x_data, y_data, epochs=5000)

스트리밍 출력 내용이 길어서 마지막 5000줄이 삭제되었습니다.
1/1 [==============================] - 0s 12ms/step - loss: 0.2340 - accuracy: 1.0000
Epoch 2502/5000
1/1 [==============================] - 0s 12ms/step - loss: 0.2337 - accuracy: 1.0000
Epoch 2503/5000
1/1 [==============================] - 0s 15ms/step - loss: 0.2335 - accuracy: 1.0000
Epoch 2504/5000
1/1 [==============================] - 0s 17ms/step - loss: 0.2332 - accuracy: 1.0000
Epoch 2505/5000
1/1 [==============================] - 0s 12ms/step - loss: 0.2329 - accuracy: 1.0000
Epoch 2506/5000
1/1 [==============================] - 0s 10ms/step - loss: 0.2326 - accuracy: 1.0000
Epoch 2507/5000
1/1 [==============================] - 0s 16ms/step - loss: 0.2324 - accuracy: 1.0000
Epoch 2508/5000
1/1 [==============================] - 0s 13ms/step - loss: 0.2321 - accuracy: 1.0000
Epoch 2509/5000
1/1 [==============================] - 0s 11ms/step - loss: 0.2318 - accuracy: 1.0000
Epoch 2510/5000
1/1 [==============================] - 0s 10ms

In [ ]:
predictions = model.predict(x_data)
print(tf.cast(predictions > 0.5, 'float32'))

score = model.evaluate(x_data, y_data)
score

tf.Tensor(
[[0.]
 [1.]
 [0.]
 [1.]], shape=(4, 1), dtype=float32)
1/1 [==============================] - 0s 55ms/step - loss: 0.6902 - accuracy: 0.5000


[0.6901970505714417, 0.5]

In [ ]:
history.history.values()

dict_values([[0.6901970505714417, 0.6901893615722656, 0.6901817321777344, 0.6901741027832031, 0.6901663541793823, 0.6901587247848511, 0.6901509761810303, 0.6901432275772095, 0.6901354789733887, 0.6901277303695679, 0.6901199221611023, 0.6901121735572815, 0.6901043057441711, 0.6900964379310608, 0.6900885701179504, 0.6900806427001953, 0.690072774887085, 0.6900649070739746, 0.6900569200515747, 0.6900490522384644, 0.6900410652160645, 0.6900330185890198, 0.6900250315666199, 0.69001704454422, 0.6900089979171753, 0.6900008916854858, 0.6899927854537964, 0.6899847388267517, 0.689976692199707, 0.6899685263633728, 0.6899603605270386, 0.6899521946907043, 0.6899439692497253, 0.6899357438087463, 0.6899275779724121, 0.6899192333221436, 0.6899110078811646, 0.6899027228355408, 0.689894437789917, 0.6898861527442932, 0.6898777484893799, 0.6898694038391113, 0.6898610591888428, 0.6898525953292847, 0.6898441910743713, 0.689835786819458, 0.6898273229598999, 0.6898188591003418, 0.6898102760314941, 0.6898018121

## 깊은 NN 해보기

레이어를 4개, 유닛을 10개로 만든다.

In [ ]:
tf.random.set_seed(1)

W1 = tf.Variable(tf.random.normal([2, 10]))  # (입력, 출력) 출력이 다음 레이어의 입력이다.
b1 = tf.Variable(tf.random.normal([10]))    

W2 = tf.Variable(tf.random.normal([10, 10])) 
b2 = tf.Variable(tf.random.normal([10]))

W3 = tf.Variable(tf.random.normal([10, 10])) 
b3 = tf.Variable(tf.random.normal([10]))

W4 = tf.Variable(tf.random.normal([10, 1])) 
b4 = tf.Variable(tf.random.normal([1]))

learning_rate = 0.1
for i in range(2001):
    with tf.GradientTape() as tape:
        layer1 = tf.sigmoid(tf.matmul(x_data, W1) + b1)       # 레이어 1
        layer2 = tf.sigmoid(tf.matmul(layer1, W2) + b2)       # 레이어 2
        layer3 = tf.sigmoid(tf.matmul(layer2, W3) + b3)       # 레이어 3
        hypothesis = tf.sigmoid(tf.matmul(layer3, W4) + b4)   # 레이어 4
        cost = -tf.reduce_mean(y_data * tf.math.log(hypothesis) + (1 - y_data) * tf.math.log(1 - hypothesis))

    W1_grad, b1_grad, W2_grad, b2_grad, W3_grad, b3_grad, W4_grad, b4_grad = tape.gradient(cost, 
                                                                                           [W1, b1, W2, b2, W3, b3, W4, b4])

    W1.assign_sub(learning_rate * W1_grad)
    b1.assign_sub(learning_rate * b1_grad)
    W2.assign_sub(learning_rate * W2_grad)
    b2.assign_sub(learning_rate * b2_grad)
    W3.assign_sub(learning_rate * W3_grad)
    b3.assign_sub(learning_rate * b3_grad)
    W4.assign_sub(learning_rate * W4_grad)
    b4.assign_sub(learning_rate * b4_grad)

    predicted = tf.cast(hypothesis > 0.5, tf.float32)
    accuracy = tf.reduce_mean(tf.cast(tf.equal(predicted, y_data), tf.float32))

    if i % 200 == 0:
        print(f"{i} \n{hypothesis} \n{cost} \n{predicted} \n{accuracy}")

0 
[[0.4476263 ]
 [0.5370337 ]
 [0.343787  ]
 [0.44152403]] 
0.7163754105567932 
[[0.]
 [1.]
 [0.]
 [0.]] 
0.75
200 
[[0.4856163 ]
 [0.5648917 ]
 [0.44626194]
 [0.5057325 ]] 
0.6868586540222168 
[[0.]
 [1.]
 [0.]
 [1.]] 
0.5
400 
[[0.46040377]
 [0.56154203]
 [0.46621302]
 [0.51691294]] 
0.6711684465408325 
[[0.]
 [1.]
 [0.]
 [1.]] 
0.5
600 
[[0.41866392]
 [0.57676077]
 [0.48046172]
 [0.53220886]] 
0.6463737487792969 
[[0.]
 [1.]
 [0.]
 [1.]] 
0.5
800 
[[0.34990862]
 [0.6195874 ]
 [0.49517527]
 [0.5465666 ]] 
0.6007735133171082 
[[0.]
 [1.]
 [0.]
 [1.]] 
0.5
1000 
[[0.25904518]
 [0.6964176 ]
 [0.511682  ]
 [0.54340804]] 
0.5289096236228943 
[[0.]
 [1.]
 [1.]
 [1.]] 
0.75
1200 
[[0.18248457]
 [0.78867453]
 [0.53980935]
 [0.49193054]] 
0.4331408739089966 
[[0.]
 [1.]
 [1.]
 [0.]] 
1.0
1400 
[[0.13893692]
 [0.85986364]
 [0.6368215 ]
 [0.36706567]] 
0.3023058772087097 
[[0.]
 [1.]
 [1.]
 [0.]] 
1.0
1600 
[[0.10515906]
 [0.9042728 ]
 [0.78897625]
 [0.2053973 ]] 
0.16966639459133148 
[[0.]
 [

### 모델로 해보기

In [ ]:
model = keras.Sequential(
    keras.layers.Dense(units=10, input_dim=2,activation='sigmoid')
)
model.add(keras.layers.Dense(units=10, activation='sigmoid'))
model.add(keras.layers.Dense(units=10, activation='sigmoid'))
model.add(keras.layers.Dense(units=1, activation='sigmoid'))

model.compile(loss='binary_crossentropy', 
              optimizer=tf.optimizers.SGD(lr=0.01), 
              metrics=['accuracy'])
model.summary()

Model: "sequential_6"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 dense_20 (Dense)            (None, 10)                30        
                                                                 
 dense_21 (Dense)            (None, 10)                110       
                                                                 
 dense_22 (Dense)            (None, 10)                110       
                                                                 
 dense_23 (Dense)            (None, 1)                 11        
                                                                 
Total params: 261
Trainable params: 261
Non-trainable params: 0
_________________________________________________________________


/usr/local/lib/python3.7/dist-packages/keras/optimizer_v2/gradient_descent.py:102: UserWarning: The `lr` argument is deprecated, use `learning_rate` instead.
  super(SGD, self).__init__(name, **kwargs)


In [ ]:
history = model.fit(x_data, y_data, epochs=1000)

Epoch 1/1000
1/1 [==============================] - 0s 479ms/step - loss: 0.6995 - accuracy: 0.5000
Epoch 2/1000
1/1 [==============================] - 0s 14ms/step - loss: 0.6994 - accuracy: 0.5000
Epoch 3/1000
1/1 [==============================] - 0s 13ms/step - loss: 0.6993 - accuracy: 0.5000
Epoch 4/1000
1/1 [==============================] - 0s 13ms/step - loss: 0.6991 - accuracy: 0.5000
Epoch 5/1000
1/1 [==============================] - 0s 9ms/step - loss: 0.6990 - accuracy: 0.5000
Epoch 6/1000
1/1 [==============================] - 0s 12ms/step - loss: 0.6989 - accuracy: 0.5000
Epoch 7/1000
1/1 [==============================] - 0s 12ms/step - loss: 0.6988 - accuracy: 0.5000
Epoch 8/1000
1/1 [==============================] - 0s 9ms/step - loss: 0.6987 - accuracy: 0.5000
Epoch 9/1000
1/1 [==============================] - 0s 9ms/step - loss: 0.6986 - accuracy: 0.5000
Epoch 10/1000
1/1 [==============================] - 0s 9ms/step - loss: 0.6985 - accuracy: 0.5000
Epoch 11/100

KeyboardInterrupt: ignored

학습이 안된다. 어디서 잘 못된 건지 아직은 잘 모르겠다. 

# mnist 데이터로 Deep wide NN 모델 만들어보기

In [2]:
# Lab 9 XOR
import datetime
import numpy as np
import os
import tensorflow as tf

x_data = np.array([[0, 0], [0, 1], [1, 0], [1, 1]], dtype=np.float32)
y_data = np.array([[0], [1], [1], [0]], dtype=np.float32)

tf.model = tf.keras.Sequential()
tf.model.add(tf.keras.layers.Dense(units=2, input_dim=2))
tf.model.add(tf.keras.layers.Activation('sigmoid'))
tf.model.add(tf.keras.layers.Dense(units=1, input_dim=2))
tf.model.add(tf.keras.layers.Activation('sigmoid'))
tf.model.compile(loss='binary_crossentropy', optimizer=tf.optimizers.SGD(lr=0.1),  metrics=['accuracy'])
tf.model.summary()

# prepare callback
log_dir = os.path.join(".", "logs", "fit", datetime.datetime.now().strftime("%Y%m%d-%H%M%S"))
tensorboard_callback = tf.keras.callbacks.TensorBoard(log_dir=log_dir, histogram_freq=1)

# add callback param to fit()
history = tf.model.fit(x_data, y_data, epochs=1000, callbacks=[tensorboard_callback])

predictions = tf.model.predict(x_data)
print('Prediction: \n', predictions)

score = tf.model.evaluate(x_data, y_data)
print('Accuracy: ', score[1])

'''
at the end of the run, open terminal / command window
cd to the source directory
tensorboard --logdir logs/fit

read more on tensorboard: https://www.tensorflow.org/tensorboard/get_started
'''

Model: "sequential_1"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 dense_2 (Dense)             (None, 2)                 6         
                                                                 
 activation_2 (Activation)   (None, 2)                 0         
                                                                 
 dense_3 (Dense)             (None, 1)                 3         
                                                                 
 activation_3 (Activation)   (None, 1)                 0         
                                                                 
Total params: 9
Trainable params: 9
Non-trainable params: 0
_________________________________________________________________
Epoch 1/1000


/usr/local/lib/python3.7/dist-packages/keras/optimizer_v2/gradient_descent.py:102: UserWarning: The `lr` argument is deprecated, use `learning_rate` instead.
  super(SGD, self).__init__(name, **kwargs)


1/1 [==============================] - 1s 630ms/step - loss: 0.8199 - accuracy: 0.5000
Epoch 2/1000
1/1 [==============================] - 0s 43ms/step - loss: 0.8104 - accuracy: 0.5000
Epoch 3/1000
1/1 [==============================] - 0s 52ms/step - loss: 0.8015 - accuracy: 0.5000
Epoch 4/1000
1/1 [==============================] - 0s 61ms/step - loss: 0.7932 - accuracy: 0.5000
Epoch 5/1000
1/1 [==============================] - 0s 41ms/step - loss: 0.7856 - accuracy: 0.5000
Epoch 6/1000
1/1 [==============================] - 0s 37ms/step - loss: 0.7785 - accuracy: 0.5000
Epoch 7/1000
1/1 [==============================] - 0s 57ms/step - loss: 0.7719 - accuracy: 0.5000
Epoch 8/1000
1/1 [==============================] - 0s 57ms/step - loss: 0.7658 - accuracy: 0.5000
Epoch 9/1000
1/1 [==============================] - 0s 45ms/step - loss: 0.7602 - accuracy: 0.5000
Epoch 10/1000
1/1 [==============================] - 0s 66ms/step - loss: 0.7550 - accuracy: 0.5000
Epoch 11/1000
1/1 [==

'\nat the end of the run, open terminal / command window\ncd to the source directory\ntensorboard --logdir logs/fit\n\nread more on tensorboard: https://www.tensorflow.org/tensorboard/get_started\n'